# 02. Baseline Models - Keyword Rule & Logistic Regression

This notebook evaluates two non-neural baseline models on the validation set:
1. **Rule-Based Keyword Matching**: Flags clauses containing common unfair phrases (`sole discretion`, `without notice`, `arbitration`, etc.).
2. **TF-IDF + Logistic Regression**: Statistical unigram/bigram n-gram model with balanced class weighting.

We compare Precision, Recall, F1-score on the unfair class, and Precision-Recall AUC (PR-AUC).


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
import joblib

# Load validation data and trained baseline models
val_df = pd.read_csv('../data/processed/val.csv')
X_val_text = val_df['text'].fillna('')
y_val = val_df['label'].values

vectorizer = joblib.load('../models/tfidf_vectorizer_baseline.joblib')
lr_model = joblib.load('../models/logreg_baseline.joblib')

print(f'Validation set size: {len(val_df)} clauses (Unfair: {y_val.sum()}, Fair: {(y_val==0).sum()})')


## 1. Inspect Logistic Regression Top Informative N-Grams
Which n-grams most strongly push a clause toward being flagged as unfair or fair?

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = lr_model.coef_[0]

top_positive_idx = np.argsort(coefs)[-15:][::-1]
top_negative_idx = np.argsort(coefs)[:15]

top_positive = pd.DataFrame({'N-gram': feature_names[top_positive_idx], 'Weight (Unfair)': coefs[top_positive_idx].round(3)})
top_negative = pd.DataFrame({'N-gram': feature_names[top_negative_idx], 'Weight (Fair)': coefs[top_negative_idx].round(3)})

print('=== TOP 15 INDICATORS OF UNFAIR CLAUSES ===')
print(top_positive.to_string(index=False))

print('\n=== TOP 15 INDICATORS OF FAIR CLAUSES ===')
print(top_negative.to_string(index=False))


## 2. Baseline Performance Summary
Read the summary results generated by `scripts/run_baselines.py`.

In [ ]:
with open('../reports/baselines_summary.md', 'r') as f:
    print(f.read())
